# 02. Random split: ProtT5-XL and structure arms

Kinase-domain PDBs (`--pdb-set kinase_domain`). Part A trains the ProtT5-XL arm and Part B the structure arm: five folds, then one full-data model trained for the median best epoch. The summary at the end reads the stored result files of all four arms (structure, ESM C 300M, ESM C 600M, ProtT5-XL).

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Config

In [ ]:
import os
from glob import glob

DRIVE_ROOT = "/content/drive/MyDrive/Diplomska_v2"
if not os.path.isdir(DRIVE_ROOT):
    shared = glob("/content/drive/.shortcut-targets-by-id/*/Diplomska_v2")
    if shared:
        DRIVE_ROOT = shared[0]
assert os.path.isdir(DRIVE_ROOT), f"Folder not found: {DRIVE_ROOT}"

PDB_SET = "kinase_domain"
PLM_MODEL = "prot_t5_xl_uniref50"
STOP_AFTER_FOLDS = None

os.environ["DTA_DATA_ROOT"] = DRIVE_ROOT
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
print("DTA_DATA_ROOT:", os.environ["DTA_DATA_ROOT"])
print("Structure set:", PDB_SET, "| language model:", PLM_MODEL)

DTA_DATA_ROOT: /content/drive/MyDrive/Diplomska_v2
Structure set: kinase_domain | language model: prot_t5_xl_uniref50


## 3. GPU

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi --query-gpu=name,memory.total --format=csv

Torch version: 2.11.0+cu130
CUDA available: True
name, memory.total [MiB]
Tesla T4, 15360 MiB


## 4. Install packages
`esm` is the EvolutionaryScale package that provides ESM C (not `fair-esm`). `biopython` is needed because the embedding script reads the PDB files.

In [ ]:
!pip install -q torch_geometric biopython esm

## 5. Go to the folder

In [ ]:
%cd {DRIVE_ROOT}
!pwd

/content/drive/.shortcut-targets-by-id/1N088IPJAiucR3uoU-A7ezJZQCcXNDgzY/Diplomska_v2
/content/drive/.shortcut-targets-by-id/1N088IPJAiucR3uoU-A7ezJZQCcXNDgzY/Diplomska_v2


## 6. Check the uploaded files

In [ ]:
import argparse, json
import pandas as pd

required = [
    "Structure_Sets.py", "Paths.py", "RunTag.py", "Conditions.py", "Protein_Embeddings.py",
    "compute_protein_embeddings.py", "Protein_Representation.py", "Creating_Train_and_Test_set.py",
    "GNNNet.py", "PLMNet.py", "DTADataset.py", "Checkpointing.py", "Seeding.py", "Emetrics.py",
    "training_5_folds.py", "testing.py", "report_best_epochs.py",
]
missing = [f for f in required if not os.path.exists(f)]
print("Missing code files:", missing or "none")

from RunTag import add_run_args, run_tag
from Seeding import SEED
parser = argparse.ArgumentParser()
add_run_args(parser)
print("--pdb-set flag accepts", PDB_SET + ":", PDB_SET in parser.format_help())

from Paths import paths_for
PATHS = paths_for("davis", PDB_SET)
assert PATHS.root == DRIVE_ROOT, f"Data root {PATHS.root} is not {DRIVE_ROOT}"
for label, path in [("pair table", PATHS.csv), ("train folds", PATHS.train_folds), ("test fold", PATHS.test_fold)]:
    print(f"{label}: {os.path.exists(path)}  {path}")

names = pd.read_csv(PATHS.csv)["protein"].unique()
pdb_missing = [n for n in names if not os.path.exists(PATHS.pdb_file(n))]
print(f"proteins in pair table: {len(names)} (expected 442)")
print(f"PDB files missing for {len(pdb_missing)} proteins", pdb_missing[:3] or "", "<- expected 0")
print("PDB folder:", PATHS.pdb)

graph_files = glob(os.path.join(PATHS.protein_graph_gml, "*_contact_map_graph.gml"))
graph_names = {os.path.basename(f)[: -len("_contact_map_graph.gml")] for f in graph_files}
print(f"protein graphs: {len(graph_files)} (expected 442), cover all proteins: {set(names) <= graph_names}")
print("protein graph folder:", PATHS.protein_graph_gml)

ligand_files = glob(os.path.join(PATHS.ligand_graphs, "*.gml"))
print(f"ligand graphs: {len(ligand_files)} (expected 68)")

with open(PATHS.train_folds) as f:
    folds = json.load(f)
print("fold sizes:", [len(x) for x in folds], "(expected 5010, 5009, 5009, 5009, 5009)")

TAG_PLM = run_tag("plm", PLM_MODEL, SEED, False, PDB_SET)
TAG_STRUCTURE = run_tag("structure", None, SEED, False, PDB_SET)
print("Run tags:", TAG_PLM, "|", TAG_STRUCTURE, "(both must end with pdb2)")
assert TAG_PLM.endswith("pdb2") and TAG_STRUCTURE.endswith("pdb2")

stale = [p for p in glob(os.path.join(PATHS.models, "*")) + glob(os.path.join(PATHS.results, "*"))
         if "pdb2" not in os.path.basename(p)]
print("Files in models/results without the pdb2 tag:", stale or "none")

Missing code files: none
--pdb-set flag accepts kinase_domain: True
pair table: True  /content/drive/MyDrive/Diplomska_v2/data/davis/davis.csv
train folds: True  /content/drive/MyDrive/Diplomska_v2/data/davis/train_folds.txt
test fold: True  /content/drive/MyDrive/Diplomska_v2/data/davis/test_fold.txt
proteins in pair table: 442 (expected 442)
PDB files missing for 0 proteins  <- expected 0
PDB folder: /content/drive/MyDrive/Diplomska_v2/data/davis/pdb_kinase_domain
protein graphs: 442 (expected 442), cover all proteins: True
protein graph folder: /content/drive/MyDrive/Diplomska_v2/data/davis/protein_graphs_kinase_domain/gml
ligand graphs: 68 (expected 68)
fold sizes: [5010, 5009, 5009, 5009, 5009] (expected 5010, 5009, 5009, 5009, 5009)
Run tags: prot_t5_xl_uniref50_pdb2 | pdb2 (both must end with pdb2)
Files in models/results without the pdb2 tag: none


## 7. Helpers

In [ ]:
import re, statistics, subprocess

QUIET_PREFIXES = ("Train epoch:", "Training on ", "Predicting for validation data",
                  "MSE improved at epoch", "No improvement since epoch")
VALIDATION_LINE = re.compile(r"^Epoch (\d+) - Validation MSE")

def run(args, stop_marker=None, every=10):
    cmd = ["python", "-u"] + args
    print("Running:", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    stopped = False
    try:
        for line in proc.stdout:
            match = VALIDATION_LINE.match(line)
            if match:
                if int(match.group(1)) % every == 0:
                    print(line, end="")
                continue
            if line.startswith(QUIET_PREFIXES):
                continue
            print(line, end="")
            if stop_marker and stop_marker in line:
                stopped = True
                print("\nStopping here as requested. Rerun this cell to continue; finished folds are skipped.")
                break
    finally:
        if proc.poll() is None:
            proc.terminate()
        proc.wait()
    if not stopped and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}, see the output above.")

def arm_flags(protein_repr):
    flags = ["--protein-repr", protein_repr, "--pdb-set", PDB_SET]
    if protein_repr == "plm":
        flags += ["--plm-model", PLM_MODEL]
    return flags

FLAGS_PLM = " ".join(arm_flags("plm"))
FLAGS_STRUCTURE = " ".join(arm_flags("structure"))

def stop_marker():
    return f"Training for fold {STOP_AFTER_FOLDS}..." if STOP_AFTER_FOLDS else None

def median_best_epoch(tag):
    path = os.path.join(PATHS.results, f"best_epochs_{tag}.json")
    with open(path) as f:
        records = json.load(f)
    folds = [str(i) for i in range(5)]
    absent = [i for i in folds if i not in records]
    assert not absent, f"Folds {absent} are not finished yet in {path}"
    print(f"{'fold':>4} {'best_epoch':>10} {'last_epoch':>10} {'best_val_mse':>13}")
    for i in folds:
        r = records[i]
        print(f"{i:>4} {r['best_epoch']:>10} {r['last_epoch']:>10} {r['best_val_mse']:>13.4f}")
    median = int(round(statistics.median(records[i]["best_epoch"] for i in folds)))
    print("Median best epoch (use for the full-data model):", median)
    return median

def show_result(model_name, tag, full=False):
    path = os.path.join(PATHS.results, f"result_{model_name}_davis_{tag}{'_full' if full else ''}.txt")
    print(path)
    with open(path) as f:
        print(f.read())

# Part A: ProtT5-XL arm

## A1. Embed the proteins

In [ ]:
!python compute_protein_embeddings.py --model {PLM_MODEL} --pdb-set {PDB_SET}

Structures: kinase_domain (/content/drive/MyDrive/Diplomska_v2/data/davis/pdb_kinase_domain)
442 proteins, 428 unique sequences, 0 to embed with prot_t5_xl_uniref50
prot_t5_xl_uniref50 pooled vectors over 428 unique sequences: mean -0.0011, std 0.0391, max |x| 0.2749, per-dimension std min 0.0024 max 0.0431


In [ ]:
cache_dir = PATHS.protein_embeddings
with open(os.path.join(cache_dir, "sequences_pdb.json")) as f:
    sequences = json.load(f)["sequences"]
unique = len(set(sequences.values()))
pooled = glob(os.path.join(cache_dir, PLM_MODEL, "*.pooled.pt"))
print("proteins:", len(sequences), "| unique sequences:", unique, "| pooled files:", len(pooled))
print("OK" if len(pooled) == unique else "MISMATCH: pooled files do not match unique sequences")
print("embedding folder:", cache_dir)
with open(os.path.join(cache_dir, PLM_MODEL, "windowed_pdb.json")) as f:
    print("windowed:", json.load(f))

proteins: 442 | unique sequences: 428 | pooled files: 428
OK
embedding folder: /content/drive/MyDrive/Diplomska_v2/data/davis/protein_embeddings_kinase_domain
windowed: {'lengths': {}, 'max_residues': None}


## A2. Train the five folds

In [ ]:
run(["training_5_folds.py"] + arm_flags("plm"), stop_marker=stop_marker())

Running: python -u training_5_folds.py --protein-repr plm --pdb-set kinase_domain --plm-model prot_t5_xl_uniref50
Dataset:  davis
Mode:  cv
Protein representation:  plm
Structure set:  kinase_domain
Protein language model:  prot_t5_xl_uniref50
Standardized embeddings:  False
Seed:  42
Learning rate:  0.001
Epochs:  2000
Early stop patience:  100
Data root:  /content/drive/MyDrive/Diplomska_v2
Using device: cuda
Run tag:  prot_t5_xl_uniref50_pdb2
Loaded prot_t5_xl_uniref50 embeddings for 442 proteins, dim 1024
Training for fold 0...
/content/drive/.shortcut-targets-by-id/1N088IPJAiucR3uoU-A7ezJZQCcXNDgzY/Diplomska_v2/training_5_folds.py:39: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  return DataLoader(
/content/drive/.shortcut-targets-by-id/1N088IPJAiucR3uoU-A7ezJZQCcXNDgzY/Diplomska_v2/training_5_folds.py:111: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  valid_loader = DataLoader(
Make prediction for 5010 samples...

## A3. Best epochs and median

In [ ]:
!python report_best_epochs.py {FLAGS_PLM}
FULL_EPOCHS_PLM = median_best_epoch(TAG_PLM)

Checkpoints: /content/drive/MyDrive/Diplomska_v2/models_davis
fold  last_epoch  best_epoch    best_mse
   0         900         800      0.1739
   1         790         694      0.1705
   2         650         556      0.1873
   3         530         430      0.1719
   4         830         734      0.1694
median best_epoch: 694
fold best_epoch last_epoch  best_val_mse
   0        800        900        0.1739
   1        694        794        0.1705
   2        556        656        0.1873
   3        430        530        0.1719
   4        734        834        0.1694
Median best epoch (use for the full-data model): 694


## A4. Score the five fold models on the test set

In [ ]:
!python testing.py {FLAGS_PLM}

Loaded prot_t5_xl_uniref50 embeddings for 442 proteins, dim 1024
dataset: davis
device: cuda:0
seed: 42
structure set: kinase_domain
Make prediction for 5010 samples...
Fold 0 metrics:
  rmse: 0.43442677045273576
  mse: 0.18872661888599396
  pearson: 0.8755724819742176
  spearman: 0.7147243787938389
  ci: 0.9018825218054921
  cindex: 0.9047625445816744
  rm2: 0.7231924533843994
/content/drive/.shortcut-targets-by-id/1N088IPJAiucR3uoU-A7ezJZQCcXNDgzY/Diplomska_v2/testing.py:89: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(loc=0, numpoints=1)
Make prediction for 5010 samples...
Fold 1 metrics:
  rmse: 0.4372005800440536
  mse: 0.19114434719085693
  pearson: 0.8747572875646887
  spearman: 0.7120631303284
  ci: 0.9000336629535827
  cindex: 0.900307033139956
  rm2: 0.7107062935829163
Make prediction for 5010 samples...
Fold 2 metrics:
  rmse: 0.4500251829

## A5. Full-data model

In [ ]:
print("Epochs:", FULL_EPOCHS_PLM)
run(["training_5_folds.py"] + arm_flags("plm") + ["--mode", "full", "--epochs", str(FULL_EPOCHS_PLM)], every=50)

Epochs: 694
Running: python -u training_5_folds.py --protein-repr plm --pdb-set kinase_domain --plm-model prot_t5_xl_uniref50 --mode full --epochs 694
Dataset:  davis
Mode:  full
Protein representation:  plm
Structure set:  kinase_domain
Protein language model:  prot_t5_xl_uniref50
Standardized embeddings:  False
Seed:  42
Learning rate:  0.001
Epochs:  694
Data root:  /content/drive/MyDrive/Diplomska_v2
Using device: cuda
Run tag:  prot_t5_xl_uniref50_pdb2
Loaded prot_t5_xl_uniref50 embeddings for 442 proteins, dim 1024
/content/drive/.shortcut-targets-by-id/1N088IPJAiucR3uoU-A7ezJZQCcXNDgzY/Diplomska_v2/training_5_folds.py:39: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  return DataLoader(
Saved full-data model after 694 epochs to /content/drive/MyDrive/Diplomska_v2/models_davis/model_PLMNet_davis_prot_t5_xl_uniref50_pdb2_full.model


In [ ]:
!python testing.py {FLAGS_PLM} --full

Loaded prot_t5_xl_uniref50 embeddings for 442 proteins, dim 1024
dataset: davis
device: cuda:0
seed: 42
structure set: kinase_domain
Make prediction for 5010 samples...
Fold full metrics:
  rmse: 0.4081745776547805
  mse: 0.16660648584365845
  pearson: 0.8910116960306084
  spearman: 0.7273392991051241
  ci: 0.9109637084873985
  cindex: 0.9098389341519217
  rm2: 0.7523004412651062
/content/drive/.shortcut-targets-by-id/1N088IPJAiucR3uoU-A7ezJZQCcXNDgzY/Diplomska_v2/testing.py:89: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(loc=0, numpoints=1)

davis  folds evaluated: ['full']
ci (all pairs) is the value to compare with published numbers; cindex only counts pairs whose larger label has the larger row index.

fold full  rmse: 0.4082  mse: 0.1666  pearson: 0.8910  spearman: 0.7273  ci: 0.9110  cindex: 0.9098  rm2: 0.7523

rmse      mean: 0.4082  std: na

In [ ]:
show_result("PLMNet", TAG_PLM)
show_result("PLMNet", TAG_PLM, full=True)

/content/drive/MyDrive/Diplomska_v2/results_davis/result_PLMNet_davis_prot_t5_xl_uniref50_pdb2.txt
davis  folds evaluated: [0, 1, 2, 3, 4]
ci (all pairs) is the value to compare with published numbers; cindex only counts pairs whose larger label has the larger row index.

fold 0  rmse: 0.4344  mse: 0.1887  pearson: 0.8756  spearman: 0.7147  ci: 0.9019  cindex: 0.9048  rm2: 0.7232
fold 1  rmse: 0.4372  mse: 0.1911  pearson: 0.8748  spearman: 0.7121  ci: 0.9000  cindex: 0.9003  rm2: 0.7107
fold 2  rmse: 0.4500  mse: 0.2025  pearson: 0.8664  spearman: 0.7129  ci: 0.8995  cindex: 0.8998  rm2: 0.7010
fold 3  rmse: 0.4470  mse: 0.1998  pearson: 0.8669  spearman: 0.7085  ci: 0.8968  cindex: 0.8977  rm2: 0.7236
fold 4  rmse: 0.4444  mse: 0.1975  pearson: 0.8698  spearman: 0.7096  ci: 0.8988  cindex: 0.8951  rm2: 0.7054

rmse      mean: 0.4426  std: 0.0066
mse       mean: 0.1959  std: 0.0058
pearson   mean: 0.8707  std: 0.0043
spearman  mean: 0.7116  std: 0.0025
ci        mean: 0.8994  std: 0.0

# Part B: Structure baseline

## B1. Train the five folds

In [ ]:
run(["training_5_folds.py"] + arm_flags("structure"), stop_marker=stop_marker())

Running: python -u training_5_folds.py --protein-repr structure --pdb-set kinase_domain
Dataset:  davis
Mode:  cv
Protein representation:  structure
Structure set:  kinase_domain
Seed:  42
Learning rate:  0.001
Epochs:  2000
Early stop patience:  100
Data root:  /content/drive/MyDrive/Diplomska_v2
Using device: cuda
Run tag:  pdb2
GNNNet Loaded
Fold 0 already finished at epoch 1120 (best MSE 0.18566997349262238 at epoch 1020), skipping
GNNNet Loaded
Resuming fold 1 at epoch 791 (best MSE 0.18206201493740082 at epoch 692)
Training for fold 1...
/content/drive/.shortcut-targets-by-id/1N088IPJAiucR3uoU-A7ezJZQCcXNDgzY/Diplomska_v2/training_5_folds.py:39: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  return DataLoader(
/content/drive/.shortcut-targets-by-id/1N088IPJAiucR3uoU-A7ezJZQCcXNDgzY/Diplomska_v2/training_5_folds.py:111: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  valid_loader = DataLoader(
Make prediction for 500

## B2. Best epochs and median

In [ ]:
!python report_best_epochs.py {FLAGS_STRUCTURE}
FULL_EPOCHS_STRUCTURE = median_best_epoch(TAG_STRUCTURE)

Checkpoints: /content/drive/MyDrive/Diplomska_v2/models_davis
fold  last_epoch  best_epoch    best_mse
   0        1120        1020      0.1857
   1         790         692      0.1821
   2        1080         982      0.1873
   3         840         742      0.1754
   4         900         804      0.1779
median best_epoch: 804
fold best_epoch last_epoch  best_val_mse
   0       1020       1120        0.1857
   1        692        792        0.1821
   2        982       1082        0.1873
   3        742        842        0.1754
   4        804        904        0.1779
Median best epoch (use for the full-data model): 804


## B3. Score the five fold models on the test set

In [ ]:
!python testing.py {FLAGS_STRUCTURE}

dataset: davis
device: cuda:0
seed: 42
structure set: kinase_domain
Test dataset created with 5010 samples.
GNNNet Loaded
Make prediction for 5010 samples...
Fold 0 metrics:
  rmse: 0.4368663524759621
  mse: 0.19085220992565155
  pearson: 0.8733153475369538
  spearman: 0.7123472364187665
  ci: 0.8994878268909414
  cindex: 0.9012795963885953
  rm2: 0.7409884929656982
/content/drive/.shortcut-targets-by-id/1N088IPJAiucR3uoU-A7ezJZQCcXNDgzY/Diplomska_v2/testing.py:89: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(loc=0, numpoints=1)
GNNNet Loaded
Make prediction for 5010 samples...
Fold 1 metrics:
  rmse: 0.44932558199147005
  mse: 0.20189347863197327
  pearson: 0.8658073310486796
  spearman: 0.7167574219398575
  ci: 0.9017513465564315
  cindex: 0.9013973976238522
  rm2: 0.7144579291343689
GNNNet Loaded
Make prediction for 5010 samples...
Fold 2 metrics:

## B4. Full-data model

In [ ]:
print("Epochs:", FULL_EPOCHS_STRUCTURE)
run(["training_5_folds.py"] + arm_flags("structure") + ["--mode", "full", "--epochs", str(FULL_EPOCHS_STRUCTURE)], every=50)

Epochs: 804
Running: python -u training_5_folds.py --protein-repr structure --pdb-set kinase_domain --mode full --epochs 804
Dataset:  davis
Mode:  full
Protein representation:  structure
Structure set:  kinase_domain
Seed:  42
Learning rate:  0.001
Epochs:  804
Data root:  /content/drive/MyDrive/Diplomska_v2
Using device: cuda
Run tag:  pdb2
GNNNet Loaded
Resuming full-data training at epoch 801
/content/drive/.shortcut-targets-by-id/1N088IPJAiucR3uoU-A7ezJZQCcXNDgzY/Diplomska_v2/training_5_folds.py:39: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  return DataLoader(
Saved full-data model after 804 epochs to /content/drive/MyDrive/Diplomska_v2/models_davis/model_GNNNet_davis_pdb2_full.model


In [ ]:
!python testing.py {FLAGS_STRUCTURE} --full

dataset: davis
device: cuda:0
seed: 42
structure set: kinase_domain
Test dataset created with 5010 samples.
GNNNet Loaded
Make prediction for 5010 samples...
Fold full metrics:
  rmse: 0.4066054183274324
  mse: 0.16532796621322632
  pearson: 0.8936443650796015
  spearman: 0.7271068814917502
  ci: 0.9104353318274343
  cindex: 0.9087024803253189
  rm2: 0.7894556522369385
/content/drive/.shortcut-targets-by-id/1N088IPJAiucR3uoU-A7ezJZQCcXNDgzY/Diplomska_v2/testing.py:89: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(loc=0, numpoints=1)

davis  folds evaluated: ['full']
ci (all pairs) is the value to compare with published numbers; cindex only counts pairs whose larger label has the larger row index.

fold full  rmse: 0.4066  mse: 0.1653  pearson: 0.8936  spearman: 0.7271  ci: 0.9104  cindex: 0.9087  rm2: 0.7895

rmse      mean: 0.4066  std: nan
mse      

In [ ]:
show_result("GNNNet", TAG_STRUCTURE)
show_result("GNNNet", TAG_STRUCTURE, full=True)

/content/drive/MyDrive/Diplomska_v2/results_davis/result_GNNNet_davis_pdb2.txt
davis  folds evaluated: [0, 1, 2, 3, 4]
ci (all pairs) is the value to compare with published numbers; cindex only counts pairs whose larger label has the larger row index.

fold 0  rmse: 0.4369  mse: 0.1909  pearson: 0.8733  spearman: 0.7123  ci: 0.8995  cindex: 0.9013  rm2: 0.7410
fold 1  rmse: 0.4493  mse: 0.2019  pearson: 0.8658  spearman: 0.7168  ci: 0.9018  cindex: 0.9014  rm2: 0.7145
fold 2  rmse: 0.4532  mse: 0.2054  pearson: 0.8633  spearman: 0.7140  ci: 0.9004  cindex: 0.9002  rm2: 0.7188
fold 3  rmse: 0.4514  mse: 0.2037  pearson: 0.8643  spearman: 0.7098  ci: 0.8977  cindex: 0.8999  rm2: 0.7181
fold 4  rmse: 0.4562  mse: 0.2082  pearson: 0.8626  spearman: 0.7070  ci: 0.8972  cindex: 0.8932  rm2: 0.7038

rmse      mean: 0.4494  std: 0.0074
mse       mean: 0.2020  std: 0.0066
pearson   mean: 0.8659  std: 0.0043
spearman  mean: 0.7120  std: 0.0038
ci        mean: 0.8993  std: 0.0019
cindex    mean: 

# Summary
Run after both parts.

In [ ]:
def mean_metric(model_name, tag, metric, full=False):
    path = os.path.join(PATHS.results, f"result_{model_name}_davis_{tag}{'_full' if full else ''}.txt")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        for line in f:
            if line.startswith(metric + " ") and "mean:" in line:
                return float(line.split("mean:")[1].split()[0])
    return None

def fmt(value):
    return f"{value:8.4f}" if value is not None else f"{'-':>8}"

MODELS = ["esmc_300m", "esmc_600m", "prot_t5_xl_uniref50"]
rows = [("structure", "GNNNet", run_tag("structure", None, SEED, False, PDB_SET))]
rows += [(m, "PLMNet", run_tag("plm", m, SEED, False, PDB_SET)) for m in MODELS]

print(f"{'run':<46} {'MSE':>8} {'CI':>8}")
for name, model_name, tag in rows:
    for full, kind in [(False, "5-fold test mean"), (True, "full data")]:
        mse = mean_metric(model_name, tag, "mse", full)
        ci = mean_metric(model_name, tag, "ci", full)
        if mse is None and ci is None:
            continue
        print(f"{'v2 ' + name + ', ' + kind:<46} {fmt(mse)} {fmt(ci)}")

run                                                 MSE       CI
v2 structure, 5-fold test mean                   0.2020   0.8993
v2 structure, full data                          0.1653   0.9104
v2 esmc_300m, 5-fold test mean                   0.1918   0.9001
v2 esmc_300m, full data                          0.1625   0.9116
v2 esmc_600m, 5-fold test mean                   0.1848   0.9012
v2 esmc_600m, full data                          0.1586   0.9075
v2 prot_t5_xl_uniref50, 5-fold test mean         0.1959   0.8994
v2 prot_t5_xl_uniref50, full data                0.1666   0.9110
